# LSTM — Previsão de Temperatura via Séries Temporais

**Objetivo:** Implementar uma Rede Neural Recorrente com células **LSTM** (Long Short-Term Memory)
para prever a temperatura máxima dos próximos dias com base em sequências históricas.

**Framework:** TensorFlow/Keras  
**Métricas:** MAE, RMSE, R²

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.data.preprocessor import get_processed_data, make_sequences

tf.random.set_seed(42)
np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)
Path('../outputs/models').mkdir(parents=True, exist_ok=True)
print(f'TensorFlow: {tf.__version__}')

## 1. Preparação da Série Temporal

In [ ]:
df = get_processed_data('../data/raw/cerrado_clima_raw.csv')

# Usa temperatura máxima diária como série alvo
series = df['temp_max'].dropna().values.reshape(-1, 1)
dates = df['temp_max'].dropna().index

# Normalização [0, 1]
scaler = MinMaxScaler()
series_scaled = scaler.fit_transform(series).flatten()

# Divisão temporal: 80% treino, 10% validação, 10% teste
n = len(series_scaled)
n_train = int(0.80 * n)
n_val   = int(0.90 * n)

N_STEPS = 30  # janela de 30 dias passados para prever o próximo

X_all, y_all = make_sequences(series_scaled, N_STEPS)

X_train = X_all[:n_train - N_STEPS]
y_train = y_all[:n_train - N_STEPS]
X_val   = X_all[n_train - N_STEPS:n_val - N_STEPS]
y_val   = y_all[n_train - N_STEPS:n_val - N_STEPS]
X_test  = X_all[n_val - N_STEPS:]
y_test  = y_all[n_val - N_STEPS:]

print(f'Série total: {n:,} dias')
print(f'Treino:  {X_train.shape} | Val: {X_val.shape} | Teste: {X_test.shape}')

## 2. Arquitetura da Rede LSTM

In [ ]:
def build_lstm(n_steps: int, units_1: int = 64, units_2: int = 32, dropout: float = 0.2):
    model = Sequential([
        LSTM(units_1, return_sequences=True, input_shape=(n_steps, 1)),
        BatchNormalization(),
        Dropout(dropout),
        LSTM(units_2, return_sequences=False),
        Dropout(dropout),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])
    return model

model = build_lstm(N_STEPS)
model.summary()

## 3. Treinamento

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ModelCheckpoint('../outputs/models/lstm_temp_max.keras',
                    monitor='val_loss', save_best_only=True, verbose=0),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

print(f'\nTreinamento concluído em {len(history.history["loss"])} épocas.')

## 4. Curvas de Aprendizado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Treino')
axes[0].plot(history.history['val_loss'], label='Validação')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Curva de Loss')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Treino')
axes[1].plot(history.history['val_mae'], label='Validação')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE')
axes[1].set_title('Curva de MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/05_lstm_learning_curves.png')
plt.show()

## 5. Avaliação no Conjunto de Teste

In [ ]:
y_pred_scaled = model.predict(X_test, verbose=0).flatten()

# Desnormaliza
y_pred_real = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

mae  = mean_absolute_error(y_test_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
r2   = r2_score(y_test_real, y_pred_real)

print(f'=== LSTM — Métricas no Conjunto de Teste ===')
print(f'MAE  = {mae:.3f} °C')
print(f'RMSE = {rmse:.3f} °C')
print(f'R²   = {r2:.4f}')

In [ ]:
test_dates = dates[n_val:]
n_plot = min(len(test_dates), len(y_test_real))
test_dates = test_dates[:n_plot]

fig, axes = plt.subplots(2, 1, figsize=(15, 9))

# Série completa de teste
axes[0].plot(test_dates, y_test_real[:n_plot], label='Real', color='steelblue', linewidth=1)
axes[0].plot(test_dates, y_pred_real[:n_plot], label='Predito (LSTM)', color='tomato',
             linewidth=1, linestyle='--')
axes[0].set_ylabel('Temperatura Máxima (°C)')
axes[0].set_title(f'LSTM — Previsão de Temperatura Máxima | MAE={mae:.2f}°C | R²={r2:.4f}')
axes[0].legend()

# Zoom: últimos 120 dias
axes[1].plot(test_dates[-120:], y_test_real[:n_plot][-120:], label='Real', color='steelblue', linewidth=2)
axes[1].plot(test_dates[-120:], y_pred_real[:n_plot][-120:], label='Predito (LSTM)',
             color='tomato', linewidth=2, linestyle='--')
axes[1].fill_between(test_dates[-120:],
                     y_test_real[:n_plot][-120:] - mae,
                     y_test_real[:n_plot][-120:] + mae,
                     alpha=0.2, color='orange', label=f'Faixa ±MAE ({mae:.2f}°C)')
axes[1].set_xlabel('Data')
axes[1].set_ylabel('Temperatura Máxima (°C)')
axes[1].set_title('Zoom — Últimos 120 dias')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/05_lstm_previsao.png')
plt.show()

In [ ]:
# Distribuição dos erros
erros = y_test_real[:n_plot] - y_pred_real[:n_plot]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(erros, bins=60, color='purple', alpha=0.7, edgecolor='black')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Erro (Real - Predito) em °C')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição dos Erros')

axes[1].scatter(y_test_real[:n_plot], y_pred_real[:n_plot], alpha=0.3, s=5, color='purple')
lim = [y_test_real.min() - 1, y_test_real.max() + 1]
axes[1].plot(lim, lim, 'r--')
axes[1].set_xlabel('Real (°C)')
axes[1].set_ylabel('Predito (°C)')
axes[1].set_title('Real vs. Predito')

plt.tight_layout()
plt.savefig('../outputs/figures/05_lstm_erros.png')
plt.show()
print(f'Erro médio: {erros.mean():.4f}°C | Desvio padrão: {erros.std():.4f}°C')

## 6. Conclusões

- A arquitetura LSTM de 2 camadas (64→32 unidades) captura adequadamente os padrões sazonais do Cerrado.
- O **Early Stopping** evitou overfitting, interrompendo o treino na época de menor val_loss.
- Com janela de 30 dias, o modelo aprende a sazonalidade e tendência de temperatura.
- **Limitação:** dados uni-variados — versão futura incorporará precipitação, umidade e radiação como features de entrada (modelo multivariado).